# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access metadata as a single object and print dataset information
metadata = dataset.metadata.to_json()
print(f"Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata.get('datePublished', 'not available')}")
print(f"Authors (by @id): {[author['@id'] for author in metadata.get('author', [])]}")
print(f"Citation: {metadata.get('citeAs', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This dataset contains multiple structured tables corresponding to clinical features, hormone levels, imaging, and genetic variant information for three patients. Each record set, field, and column is identified using an `@id`.

Let's retrieve the list of record sets and their details. All entities will be referenced by their `@id`.

In [ ]:
record_sets = []

# Iterate over record sets in dataset.metadata
recordsets_attr = getattr(dataset.metadata, 'recordSet', [])
if not recordsets_attr:
    print("No record sets found directly in metadata. Attempting alternative access...")
    # Try obtaining IDs from the Dataset JSON if not present as objects
    dataset_json = dataset.metadata.to_json()
    recordsets_attr = dataset_json.get('recordSet', [])

for recordset in recordsets_attr:
    rec_id = recordset['@id'] if isinstance(recordset, dict) else recordset
    record_sets.append(rec_id)

# If no recordSet IDs are present (empty recordSet list in metadata), we will try to infer from the schema
if not record_sets:
    # mlcroissant exposes dataset.record_sets attribute
    try:
        for rs in dataset.record_sets:
            print(f"RecordSet: {rs['@id']} -- name: {rs.get('name', '')}")
            record_sets.append(rs['@id'])
            if 'field' in rs:
                print("  Fields:")
                for f in rs['field']:
                    f_id = f['@id'] if isinstance(f, dict) else f
                    print(f"    - {f_id}")
            else:
                print("  No field info found.")
    except AttributeError:
        print("No record_sets attribute found in dataset. Please consult the Croissant JSON directly.")
else:
    print('RecordSets found in metadata:')
    for rec_id in record_sets:
        print(f"  - {rec_id}")

# For demonstration, print example records from the first record set
if record_sets:
    first_rs = record_sets[0]
    print(f"Sample records from record set {first_rs}:")
    for i, record in enumerate(dataset.records(record_set=first_rs)):
        print(record)
        if i >= 2:
            break


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview. All entities are referenced by their `@id`.

In [ ]:
# Assuming the dataset contains three main tables as described in the metadata:
# (1) Table 1—clinical features
# (2) Table 2—hormone levels and imaging
# (3) Table 3—genetic variants
# Attempt to extract these using common record set IDs (adjust as needed based on actual schema IDs)
dataframes = {}

# If record_sets is empty, try to infer from dataset.record_sets
if not record_sets:
    record_sets = [rs['@id'] for rs in dataset.record_sets]

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet {rs_id}. Columns: {df.columns.tolist()}")
        print(df.head())

# For subsequent steps, select the first available record set
if dataframes:
    selected_rs_id = list(dataframes.keys())[0]
    print(f"Selected record set: {selected_rs_id}\nColumns: {dataframes[selected_rs_id].columns.tolist()}")
else:
    selected_rs_id = None


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Transform, filter, and group based on available numeric fields and categorical attributes in the example table. All references use column `@id`.

In [ ]:
# Choose a numeric field for filtering and normalization
if selected_rs_id and selected_rs_id in dataframes:
    df = dataframes[selected_rs_id]
    # Try to identify a numeric field - use 'age_at_diagnosis' or similar
    numeric_field_id = None
    for col in df.columns:
        if 'age' in col or 'height' in col:
            numeric_field_id = col
            break
    # Fallback to first column if not found
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    print(f"Numeric field for analysis: {numeric_field_id}")

    # Filtering
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records (where {numeric_field_id} > {threshold}):\n", filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field, e.g. 'infertility' or 'patient_id', if present
    group_field_id = None
    for col in df.columns:
        if 'infertility' in col or 'hirsutism' in col or 'patient' in col:
            group_field_id = col
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped by {group_field_id} (mean of numeric fields):\n", grouped_df.head())
else:
    print("No DataFrame to analyze. Please check earlier steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll show a simple histogram and a scatter plot for numeric columns, referencing all entities by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

if selected_rs_id and selected_rs_id in dataframes and numeric_field_id:
    df = dataframes[selected_rs_id]
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist(bins=5)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot: age vs height (if both exist)
    height_field_id = None
    for col in df.columns:
        if 'height' in col:
            height_field_id = col
            break

    if numeric_field_id and height_field_id:
        plt.figure(figsize=(6, 4))
        plt.scatter(df[numeric_field_id], df[height_field_id])
        plt.title(f"Scatter: {numeric_field_id} vs {height_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(height_field_id)
        plt.show()
else:
    print("No DataFrame or numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset contains structured clinical, hormonal, imaging, and genetic data for three female patients with non-classical 21-hydroxylase deficiency-associated infertility. 
- Using `mlcroissant`, record sets and fields were identified and referenced by their `@id`.
- Main numeric features were filtered and normalized; basic exploratory plots illustrated patient-level variance.
- Due to the small cohort size, analysis provides illustrative clinical patterns but is limited by representativeness and sample size.
- The Croissant schema and `mlcroissant` facilitate reproducible FAIR data access for rare endocrine research use cases.